# Dataset Generation

This notebook generates the single authoritative synthetic dataset used by every model notebook. It is deterministic (`random_state=42`), contains 10,000 unique feature combinations, and creates the target from interpretable customer/order preferences with controlled randomness.

In [1]:
import pandas as pd
import numpy as np

rng=np.random.default_rng(42)
genders=["Male","Female","Other"]; ages=["18-24","25-34","35-44","45-54","55-64"]
marital=["Single","Married","Divorced","Widowed"]; income=["Low","Middle","Upper-Middle","High"]
times=["Breakfast","Lunch","Snack","Dinner","Late Night"]; days=["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
locations=["Benz Circle","MG Road","Governorpet","Auto Nagar","Gunadala","Patamata","Bhavanipuram","Ramachandra Nagar","One Town","Poranki"]
cuisines=["South Indian","Biryani","Chinese","North Indian","Fast Food","Bakery","Continental","Street Food"]
seen=set(); rows=[]
location_primary={"Benz Circle":"Fast Food","MG Road":"Chinese","Governorpet":"South Indian","Auto Nagar":"Biryani","Gunadala":"South Indian","Patamata":"Continental","Bhavanipuram":"Biryani","Ramachandra Nagar":"North Indian","One Town":"Street Food","Poranki":"Bakery"}
time_primary={"Breakfast":"South Indian","Lunch":"Biryani","Snack":"Street Food","Dinner":"Biryani","Late Night":"Fast Food"}
for _ in range(10000):
    while True:
        feat=(rng.choice(genders),rng.choice(ages),rng.choice(marital),rng.choice(income),rng.choice(times),rng.choice(days),rng.choice(locations))
        if feat not in seen:
            seen.add(feat)
            break
    g,a,m,inc,t,d,loc=feat
    target=location_primary[loc] if rng.random() < 0.72 else time_primary[t]
    if rng.random() < 0.10:
        target=rng.choice([c for c in cuisines if c != target])
    rows.append(feat+(target,))

df=pd.DataFrame(rows,columns=["Gender","Age_Group","Marital_Status","Income_Level","Time_of_Day","Day","Location","Cuisine_Type"])
df=df.sample(frac=1,random_state=42).reset_index(drop=True)
df.to_csv("vijayawada_food_orders_dataset.csv",index=False)
print("Generated:",df.shape)
print("Missing values:",int(df.isna().sum().sum()))
print("Duplicate rows:",int(df.duplicated().sum()))
display(df.head())
display(df["Cuisine_Type"].value_counts())

Generated: (10000, 8)
Missing values: 0
Duplicate rows: 0


,Gender,Age_Group,Marital_Status,Income_Level,Time_of_Day,Day,Location,Cuisine_Type
0,Female,45-54,Widowed,Low,Breakfast,Wednesday,Bhavanipuram,South Indian
1,Other,25-34,Widowed,Upper-Middle,Breakfast,Friday,Auto Nagar,South Indian
2,Other,35-44,Married,Low,Late Night,Thursday,Poranki,Bakery
3,Male,45-54,Single,Middle,Late Night,Friday,Ramachandra Nagar,Fast Food
4,Male,55-64,Married,Upper-Middle,Dinner,Thursday,One Town,Street Food


Cuisine_Type
Biryani         2433
South Indian    1888
Street Food     1272
Fast Food       1271
Continental      798
North Indian     789
Chinese          780
Bakery           769
Name: count, dtype: int64

## Data quality

The generated data is synthetic and intended for an educational classification project. The target is not derived from a row identifier, and the model notebooks exclude any identifier from the features.